# Find the Intruders — SIPTA Summer School Uncertainty Challenge

A batch of **1000 handwritten-character images** came out of a digit-scanning
pipeline. Most of them are ordinary digits — but **exactly 30 images do not
belong**.

**Your task:** find the 30 intruders and submit their ids (0–999) on the
challenge website.

**Rules**
- Train your own model(s) on the **standard MNIST training set** — the
  challenge images are your only test data.
- Your detection must be based on **uncertainty quantification with
  [probly](https://github.com/pwhofman/probly)** applied to your own models.
  Matching images against public datasets or picking images by eye is not a
  method.
- A submission is **up to 30 ids**, pasted into the website as a
  comma-separated list. Your score is the number of true intruders in it.

How you model the problem, what uncertainties you compute, and how you turn it into 30 ids
is entirely up to you — that is the challenge.


In [ ]:
%pip install numpy scipy scikit-learn matplotlib pandas
%pip install torch torchvision probly==0.9.1

import matplotlib.pyplot as plt  # noqa: F401
import numpy as np
import probly  # noqa: F401
import torch
import torch.nn.functional as F  # noqa: F401
from torch import nn

torch.manual_seed(0)
np.random.seed(0)
print("setup ok")


In [ ]:
# Load the challenge data. Priority: local file -> download URL -> Colab upload.
from pathlib import Path
from urllib.request import urlretrieve

DATA_URL = "https://raw.githubusercontent.com/timoverse/SIPTASummerSchoolProblyChallenge/master/challenge_data.npz"

data_path = Path("challenge_data.npz")
if not data_path.exists() and Path("artifacts/challenge/challenge_data.npz").exists():
    data_path = Path("artifacts/challenge/challenge_data.npz")
if not data_path.exists() and DATA_URL:
    print("downloading challenge_data.npz ...")
    urlretrieve(DATA_URL, data_path)
if not data_path.exists():
    try:
        from google.colab import files  # type: ignore
        print("Please upload challenge_data.npz")
        uploaded = files.upload()
        data_path = Path(next(iter(uploaded)))
    except ImportError as exc:
        raise FileNotFoundError("challenge_data.npz not found") from exc

challenge = np.load(data_path)
images = challenge["images"].astype(np.float32) / 255.0   # (1000, 28, 28) in [0, 1]
ids = challenge["ids"]                                     # 0..999
print(images.shape, images.dtype)


In [ ]:
# Standard MNIST training data (untouched — train on this)
from torchvision import datasets

mnist_root = "artifacts/mnist" if Path("artifacts/mnist").exists() else "./mnist_data"
train_set = datasets.MNIST(root=mnist_root, train=True, download=True)
train_images = train_set.data.numpy().astype(np.float32) / 255.0
train_labels = train_set.targets.numpy()
print(train_images.shape)


In [ ]:
# -- probly API crash course ---------------------------------------------
# probly follows a pipeline of steps: transform a PyTorch model into an
# *uncertainty-aware* model, build an uncertainty representation from its
# predictions, and quantify uncertainty based on that representation.
# Everything below runs as-is (with an untrained demo model) - adapt the
# pieces to your approach. Docs & examples: https://github.com/pwhofman/probly

from probly.method import ensemble, dropout  # noqa: I001
from probly.predictor import predict
from probly.representer import representer
from probly.quantification import quantify

# 1) Base model: for 28x28 MNIST a small CNN is entirely sufficient
#    (trains in about a minute on CPU).
base_module = nn.Sequential(
    nn.Conv2d(1, 16, 5), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 5), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(32 * 4 * 4, 128), nn.ReLU(), nn.Linear(128, 10))

# 2) Uncertainty-aware model: probly.method wraps the base module.
#    predictor_type declares what the model outputs - here raw class logits.
demo_members = ensemble(base_module, num_members=3,
                        predictor_type="logit_classifier")  # ModuleList of independently
                                                            # re-initialized copies - you train them
demo_mc = dropout(base_module, p=0.25,
                  predictor_type="logit_classifier")        # MC-dropout alternative

# 3) Representation: a representer evaluates the model and collects the
#    individual predictions into a "sample" - probly's input type for
#    quantification. (For a dropout model, pass num_samples=... .)
demo_x = torch.from_numpy(images[:8][:, None])              # any batch of images
with torch.no_grad():
    demo_sample = representer(demo_members).represent(demo_x)

# 4) Quantification: quantify() turns the sample into uncertainty values.
#    The result bundles several uncertainty notions as attributes - inspect it! (e.g. print(demo_q.total))
#    Further measures live in probly.quantification.measure, including
#    credal-set based ones built from the same sample via
#    probly.representation.credal_set (create_convex_credal_set,
#    create_probability_intervals).
demo_q = quantify(demo_sample)
print("quantify ->", type(demo_q).__name__)

# 5) (Optional) Raw outputs: for your own custom scores, predict() stacks the member
#    predictions; the result exposes plain tensors.
with torch.no_grad():
    demo_out = predict(demo_members, demo_x)
print("logits:", tuple(demo_out.logits.shape),
      " probabilities:", tuple(demo_out.probabilities.shape))


## Your approach

Everything between here and the submission cell is yours: model the problem,
quantify the uncertainty of your predictions on the 1000 images with probly,
and decide which 30 ids to submit.


In [ ]:
# your model(s) and training


In [ ]:
# your uncertainty quantification


In [ ]:
# from scores to 30 candidate ids


In [ ]:
# Your final answer: fill my_ids with up to 30 image ids, run this cell, and
# paste the printed line into the challenge website.
my_ids = []

assert len(my_ids) == len(set(my_ids)) <= 30
print("SUBMISSION:")
print(",".join(str(int(i)) for i in my_ids))
